# Bali Fiber AI Assistant - Data Processing Pipeline

This notebook processes customer cluster data and generates:
- Cluster documents (text descriptions)
- Vector embeddings
- ChromaDB database
- Pickle files for efficient loading

**Run this notebook once to generate all required files for the Streamlit app.**

## 1. Install Dependencies

In [ ]:
!pip install -q langchain langchain-community chromadb sentence-transformers pandas

## 2. Load Dataset from Local File

In [ ]:
import pandas as pd

# Load from local dataset folder
df = pd.read_csv(
    'dataset/data_hasil_preprocessing.csv',
    sep=',',
    encoding='utf-8'
)
df.columns = df.columns.str.strip()

print(f'Dataset shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print(f'\nFirst 5 rows:')
df.head()

## 3. Generate Cluster Documents

In [ ]:
import pickle

# List untuk menyimpan dokumen cluster
cluster_docs = []
cluster_metadata = {}

for cluster_id in sorted(df['Cluster'].unique()):
    cluster_data = df[df['Cluster'] == cluster_id]
    
    # Informasi dominan
    dominant_segment = cluster_data['Segmentasi'].mode()[0]
    dominant_status  = cluster_data['Status'].mode()[0]
    dominant_product = cluster_data['Product'].mode()[0]
    dominant_offer   = cluster_data['offer status'].mode()[0]
    
    # Statistik numerik
    avg_bw    = cluster_data['B/W (Mbps)'].mean()
    avg_otc   = cluster_data['OTC'].mean()
    avg_mrc   = cluster_data['MRC'].mean()
    avg_total = cluster_data['GRAND TOTAL'].mean()
    total_customer = len(cluster_data)
    
    text = f"""
    Cluster {cluster_id} terdiri dari {total_customer} data pelanggan.
    Cluster ini didominasi oleh segmentasi {dominant_segment} dengan status {dominant_status}.
    Produk yang paling banyak digunakan adalah {dominant_product}.
    Mayoritas pelanggan memiliki offer status {dominant_offer}.
    Nilai rata-rata bandwidth sebesar {avg_bw:.4f} Mbps,
    rata-rata OTC sebesar {avg_otc:.4f},
    rata-rata MRC sebesar {avg_mrc:.4f},
    dan rata-rata GRAND TOTAL sebesar {avg_total:.4f}.
    Cluster ini merepresentasikan pola pelanggan berdasarkan
    aktivitas sales, produk layanan, dan potensi revenue.
    """
    
    cluster_docs.append(text)
    cluster_metadata[cluster_id] = {
        'total_customer': total_customer,
        'dominant_segment': dominant_segment,
        'dominant_status': dominant_status,
        'dominant_product': dominant_product,
        'dominant_offer': dominant_offer,
        'avg_bw': avg_bw,
        'avg_otc': avg_otc,
        'avg_mrc': avg_mrc,
        'avg_total': avg_total
    }

# Preview hasil
for i, doc in enumerate(cluster_docs):
    print(f'\n============ CLUSTER {i} ============')
    print(doc)

print(f'\n✓ Generated {len(cluster_docs)} cluster documents')

## 4. Save Cluster Documents as Pickle

In [ ]:
# Save cluster documents
with open('cluster_docs.pkl', 'wb') as f:
    pickle.dump(cluster_docs, f)
print('✓ Saved cluster_docs.pkl')

# Save cluster metadata
with open('cluster_metadata.pkl', 'wb') as f:
    pickle.dump(cluster_metadata, f)
print('✓ Saved cluster_metadata.pkl')

## 5. Create Embeddings and Vector Database

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# Inisialisasi embedding function
embedding_function = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)

print('✓ Embedding model loaded')

# Buat vector database dari dokumen cluster
vectordb = Chroma.from_texts(
    texts=cluster_docs,
    embedding=embedding_function,
    persist_directory='cluster_db'
)

doc_count = vectordb._collection.count()
print(f'✓ Vector database created with {doc_count} documents')
print(f'✓ Database saved to: cluster_db/')

## 6. Test Retrieval

In [ ]:
# Test semantic search
retriever = vectordb.as_retriever(search_kwargs={'k': 3})

test_queries = [
    'Which cluster has the highest churn risk?',
    'Cluster mana yang paling potensial?',
    'Which cluster has the most active customers?'
]

for query in test_queries:
    print(f'\n🔍 Query: {query}')
    docs = retriever.invoke(query)
    print(f'Found {len(docs)} relevant documents')
    print(f'Top result: {docs[0].page_content[:200]}...')

## 7. Summary

✅ **Processing Complete!**

Generated files:
- `cluster_docs.pkl` - Cluster text documents
- `cluster_metadata.pkl` - Cluster statistics
- `cluster_db/` - ChromaDB vector database

You can now run the Streamlit app:
```bash
streamlit run app.py
```